# Extension backend: private Colab CPU session

Load the saved BM25-top-three DistilBERT model and evidence from shared Drive.
No training, full-corpus retrieval, GPU, public tunnel or third-party web proxy is used.
This notebook contains its source and runs directly in the kernel. Requests arrive
from an authenticated CLI relay on your own computer, where Chrome still uses
`http://127.0.0.1:8000`. Backend and relay both bind only to their own loopback.

Use your paid Colab account with positive compute credits. This is a supervised
prototype on known benchmark claims, not arbitrary live claim verification or
an always-on service. Colab's normal resource and usage policies still apply.

**CLI route:** create a named CPU session, authorize Drive with `colab drivemount`,
then execute this notebook with `colab exec --timeout 600 -f ...`. See
`apps/COLAB_BACKEND.md` or `scripts/colab_extension.ps1` in the repository.
**Browser route:** select CPU/None as accelerator and run in order; use
`colab sessions` on your computer to identify this same runtime before relaying.

## 1. Restore the bundled source

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, os, subprocess, sys, time
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
SOURCES = json.loads('{"apps/backend/baseline_adapter.py": "\\"\\"\\"Adapter that runs the saved BM25-top-3 DistilBERT baseline on benchmark claims.\\n\\nReal inference, restricted to known AVeriTeC dev claims: the saved dev BM25\\nrankings supply each claim\'s three passages. Inputs are built exactly like\\nscripts/train_transformer_verifier.py::compose_text, and the model, label order\\nand token budgets come from the saved baseline run. Gold labels are never read.\\n\\"\\"\\"\\nimport json\\nfrom pathlib import Path\\n\\nLABELS = (\\"Supported\\", \\"Refuted\\", \\"Not Enough Evidence\\", \\"Conflicting Evidence\\")\\nMAX_EVIDENCE = 3\\n\\n# Standing caveat. 0.52 accuracy / 0.347 macro-F1 are the saved baseline\'s\\n# results on the 500 official dev claims (final_dev_metrics.json).\\nBASELINE_NOTE = (\\"This is a simple baseline model. On the benchmark it agreed with the \\"\\n                 \\"reference label about half the time, so treat it as a rough lead only.\\")\\n\\n\\ndef compose_text(claim, passages, tokenizer, claim_budget, evidence_budget):\\n    \\"\\"\\"Mirror of scripts/train_transformer_verifier.py::compose_text. Keep in sync.\\"\\"\\"\\n    claim_tokens = tokenizer.encode(str(claim), add_special_tokens=False)[:claim_budget]\\n    sections = [f\\"Claim: {tokenizer.decode(claim_tokens, skip_special_tokens=True)}\\"]\\n    usable = [p for p in passages[:MAX_EVIDENCE] if str(p.get(\\"text\\") or \\"\\").strip()]\\n    if not usable:\\n        sections.append(\\"Evidence: No candidate passage was retrieved.\\")\\n    for i, p in enumerate(usable, start=1):\\n        tokens = tokenizer.encode(str(p[\\"text\\"]), add_special_tokens=False)[:evidence_budget]\\n        sections.append(f\\"Evidence {i}: {tokenizer.decode(tokens, skip_special_tokens=True)}\\")\\n    return \\"\\\\n\\".join(sections)\\n\\n\\ndef load_rankings(path):\\n    rows = {}\\n    with open(path, encoding=\\"utf-8\\") as handle:\\n        for line in handle:\\n            if line.strip():\\n                row = json.loads(line)\\n                rows[str(row[\\"claim_id\\"])] = row\\n    return rows\\n\\n\\ndef lean_note(probabilities):\\n    \\"\\"\\"Plain-language strength of the model\'s lean. No percentages (not calibrated).\\"\\"\\"\\n    top = max(probabilities)\\n    if top >= 0.7:\\n        return \\"The model leaned clearly toward this label.\\"\\n    if top >= 0.45:\\n        return \\"The model leaned toward this label, but not strongly.\\"\\n    return \\"The model had no strong lean; the labels were close.\\"\\n\\n\\nclass BaselineAdapter:\\n    name = \\"baseline_distilbert_bm25_top3\\"\\n\\n    def __init__(self, rankings, predict_fn):\\n        \\"\\"\\"rankings: {claim_id: row with \'claim\' and \'retrieved\'}.\\n        predict_fn(claim, passages) -> four probabilities in LABELS order.\\"\\"\\"\\n        self.rankings = rankings\\n        self.predict_fn = predict_fn\\n        self._cache = {}\\n\\n    @classmethod\\n    def from_paths(cls, baseline_dir, rankings_path, device=\\"cpu\\"):\\n        import torch\\n        from transformers import AutoModelForSequenceClassification, AutoTokenizer\\n\\n        baseline_dir = Path(baseline_dir)\\n        settings = json.loads((baseline_dir / \\"transformer_run_config.json\\").read_text(encoding=\\"utf-8\\"))\\n        model_dir = baseline_dir / \\"final_model\\"\\n        tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, use_fast=True)\\n        model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True).to(device).eval()\\n        if [model.config.id2label.get(i) for i in range(4)] != list(LABELS):\\n            raise ValueError(\\"Saved model\'s label order differs from the canonical labels\\")\\n\\n        def predict(claim, passages):\\n            text = compose_text(claim, passages, tokenizer, settings[\\"claim_token_budget\\"],\\n                                settings[\\"evidence_token_budget_per_passage\\"])\\n            enc = tokenizer(text, truncation=True, max_length=settings[\\"max_length\\"], return_tensors=\\"pt\\")\\n            with torch.inference_mode():\\n                logits = model(**{k: v.to(device) for k, v in enc.items()}).logits.float()\\n            return logits.softmax(-1)[0].cpu().tolist()\\n\\n        return cls(load_rankings(rankings_path), predict)\\n\\n    def list_claims(self, query=\\"\\", limit=50):\\n        q = (query or \\"\\").strip().lower()\\n        out = []\\n        for cid, row in self.rankings.items():\\n            if q in cid.lower() or q in str(row.get(\\"claim\\", \\"\\")).lower():\\n                out.append({\\"claim_id\\": cid, \\"claim\\": row.get(\\"claim\\", \\"\\")})\\n                if len(out) >= limit:\\n                    break\\n        return out\\n\\n    def get(self, claim_id):\\n        row = self.rankings.get(claim_id)\\n        if row is None:\\n            return None\\n        if claim_id not in self._cache:\\n            self._cache[claim_id] = self._verify(row)\\n        return self._cache[claim_id]\\n\\n    def _verify(self, row):\\n        claim = row[\\"claim\\"]\\n        passages = row.get(\\"retrieved\\") or []\\n        evidence = [{\\"passage_id\\": str(p[\\"passage_id\\"]), \\"text\\": p[\\"text\\"],\\n                     \\"url\\": p.get(\\"url\\") or \\"\\", \\"rank\\": i}\\n                    for i, p in enumerate(passages[:MAX_EVIDENCE], start=1)\\n                    if str(p.get(\\"text\\") or \\"\\").strip()]\\n        if not evidence:\\n            return {\\"claim_id\\": row[\\"claim_id\\"], \\"claim\\": claim, \\"status\\": \\"insufficient_sources\\",\\n                    \\"explanation\\": \\"No source passages were found for this claim.\\", \\"evidence\\": []}\\n        probs = self.predict_fn(claim, passages)\\n        verdict = LABELS[max(range(4), key=lambda i: probs[i])]\\n        return {\\"claim_id\\": row[\\"claim_id\\"], \\"claim\\": claim, \\"status\\": \\"completed\\", \\"verdict\\": verdict,\\n                \\"explanation\\": \\"A baseline classifier read the claim and the three passages shown \\"\\n                               \\"and chose this label. It does not give reasons.\\",\\n                \\"uncertainty_note\\": lean_note(probs) + \\" \\" + BASELINE_NOTE,\\n                \\"evidence\\": evidence}\\n", "apps/backend/colab_runtime.py": "\\"\\"\\"Start the private benchmark backend inside a Colab CPU kernel.\\"\\"\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport importlib.util\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport shutil\\nimport threading\\nimport time\\nfrom http.server import ThreadingHTTPServer\\n\\nRUN_PATH = Path(\\"outputs/initial_ml_solution_rerun_bs16/transformer/bm25_top3\\")\\nRANKINGS_PATH = Path(\\"outputs/bm25/dev_bm25_top20_v2/dev_rankings.jsonl\\")\\n_ACTIVE = None\\n\\n\\ndef file_hash(path):\\n    digest = hashlib.sha256()\\n    with Path(path).open(\\"rb\\") as handle:\\n        for block in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(block)\\n    return digest.hexdigest()\\n\\n\\ndef stage_artifacts(data_root, cache_root):\\n    \\"\\"\\"Copy a small serving bundle atomically; never write to shared inputs.\\"\\"\\"\\n    data_root, cache_root = Path(data_root).resolve(), Path(cache_root).resolve()\\n    run = data_root / RUN_PATH\\n    model = run / \\"final_model\\"\\n    required = [run / \\"transformer_run_config.json\\", model / \\"config.json\\", data_root / RANKINGS_PATH]\\n    missing = [str(path) for path in required if not path.is_file()]\\n    if not model.is_dir() or not any((model / name).is_file() for name in (\\"model.safetensors\\", \\"pytorch_model.bin\\")):\\n        missing.append(str(model / \\"model.safetensors\\"))\\n    if missing:\\n        raise FileNotFoundError(\\"Check the mounted shared Drive and DATA_ROOT:\\\\n\\" + \\"\\\\n\\".join(missing))\\n    if cache_root == data_root or cache_root.is_relative_to(data_root):\\n        raise ValueError(\\"Use a runtime cache outside the shared input folder\\")\\n    # final_model contains the saved tokenizer and weights. Training checkpoint\\n    # subdirectories and the full evidence corpus are deliberately not copied.\\n    sources = [path for path in sorted(model.iterdir()) if path.is_file()]\\n    sources += [run / \\"transformer_run_config.json\\", data_root / RANKINGS_PATH]\\n    manifest_path = cache_root / \\"serving_manifest.json\\"\\n    previous = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}\\n    previous = previous if previous.get(\\"data_root\\") == str(data_root) else {}\\n    manifest = {\\"data_root\\": str(data_root), \\"files\\": {}}\\n    for source in sources:\\n        relative = (Path(\\"baseline/final_model\\") / source.name if source.parent == model\\n                    else Path(\\"baseline/transformer_run_config.json\\") if source.parent == run\\n                    else Path(\\"dev_rankings.jsonl\\"))\\n        target = cache_root / relative\\n        stat = source.stat()\\n        fingerprint = {\\"source\\": str(source), \\"size\\": stat.st_size, \\"mtime_ns\\": stat.st_mtime_ns}\\n        old = previous.get(\\"files\\", {}).get(relative.as_posix(), {})\\n        unchanged = all(old.get(key) == value for key, value in fingerprint.items())\\n        if unchanged and target.is_file() and file_hash(target) == old.get(\\"sha256\\"):\\n            print(\\"Reusing\\", relative, flush=True)\\n            manifest[\\"files\\"][relative.as_posix()] = old\\n            continue\\n        target.parent.mkdir(parents=True, exist_ok=True)\\n        temporary = target.with_name(target.name + \\".copying\\")\\n        print(f\\"Copying {relative} ({stat.st_size / 1e6:.1f} MB)\\", flush=True)\\n        shutil.copyfile(source, temporary)\\n        after = source.stat()\\n        if (after.st_size, after.st_mtime_ns) != (stat.st_size, stat.st_mtime_ns):\\n            raise RuntimeError(\\"Source changed during copy; rerun after it is stable: \\" + str(source))\\n        digest = file_hash(temporary)\\n        os.replace(temporary, target)\\n        manifest[\\"files\\"][relative.as_posix()] = {**fingerprint, \\"sha256\\": digest}\\n    cache_root.mkdir(parents=True, exist_ok=True)\\n    temporary = manifest_path.with_suffix(\\".json.tmp\\")\\n    temporary.write_text(json.dumps(manifest, indent=2) + \\"\\\\n\\", encoding=\\"utf-8\\")\\n    os.replace(temporary, manifest_path)\\n    return cache_root / \\"baseline\\", cache_root / \\"dev_rankings.jsonl\\", manifest\\n\\n\\ndef _module(name, filename):\\n    spec = importlib.util.spec_from_file_location(name, Path(__file__).with_name(filename))\\n    module = importlib.util.module_from_spec(spec)\\n    spec.loader.exec_module(module)\\n    return module\\n\\n\\nclass SerializedAdapter:\\n    \\"\\"\\"Keep concurrent requests from oversubscribing a small CPU runtime.\\"\\"\\"\\n    def __init__(self, adapter):\\n        self.adapter, self.name = adapter, adapter.name\\n        self.lock = threading.Lock()\\n\\n    def get(self, claim_id):\\n        with self.lock:\\n            return self.adapter.get(claim_id)\\n\\n    def list_claims(self, query=\\"\\", limit=50):\\n        return self.adapter.list_claims(query, limit)\\n\\n\\ndef start_backend(data_root, cache_root=\\"/content/claim_check_serving\\", *, port=8000,\\n                  cpu_threads=2, adapter_factory=None):\\n    \\"\\"\\"Load once, serve on runtime loopback, and return to the notebook kernel.\\n\\n    The local authenticated CLI relay is the only intended client. No public\\n    tunnel, retraining, new retrieval, or artificial runtime keepalive is used.\\n    \\"\\"\\"\\n    global _ACTIVE\\n    settings = (str(Path(data_root).resolve()), str(Path(cache_root).resolve()), port, cpu_threads)\\n    if _ACTIVE:\\n        if _ACTIVE[\\"settings\\"] != settings:\\n            raise RuntimeError(\\"Stop the existing backend before changing its settings\\")\\n        if _ACTIVE[\\"thread\\"].is_alive():\\n            return {**_ACTIVE[\\"info\\"], \\"reused\\": True}\\n        raise RuntimeError(\\"Previous backend stopped unexpectedly; call stop_backend() and restart\\")\\n    if not 0 <= port <= 65535 or not 1 <= cpu_threads <= 64:\\n        raise ValueError(\\"Invalid port or CPU thread count\\")\\n    os.environ[\\"USE_TF\\"] = \\"0\\"\\n    os.environ[\\"USE_FLAX\\"] = \\"0\\"\\n    os.environ[\\"TOKENIZERS_PARALLELISM\\"] = \\"false\\"\\n    baseline, rankings, manifest = stage_artifacts(data_root, cache_root)\\n    if adapter_factory is None:\\n        import torch\\n        torch.set_num_threads(cpu_threads)\\n        adapter_factory = _module(\\"_claim_check_baseline\\", \\"baseline_adapter.py\\").BaselineAdapter.from_paths\\n    started = time.monotonic()\\n    print(\\"Loading the saved model and rankings on CPU...\\", flush=True)\\n    adapter = SerializedAdapter(adapter_factory(baseline, rankings, \\"cpu\\"))\\n    server_module = _module(\\"_claim_check_http\\", \\"server.py\\")\\n    server = ThreadingHTTPServer((\\"127.0.0.1\\", port), server_module.make_handler(adapter))\\n    thread = threading.Thread(target=server.serve_forever, name=\\"claim-check-backend\\", daemon=True)\\n    thread.start()\\n    info = {\\"url\\": f\\"http://127.0.0.1:{server.server_port}\\", \\"adapter\\": adapter.name,\\n            \\"device\\": \\"cpu\\", \\"loaded_in_seconds\\": round(time.monotonic() - started, 2),\\n            \\"claim_count\\": len(adapter.list_claims(\\"\\", 10000)), \\"cache_root\\": str(Path(cache_root)),\\n            \\"manifest\\": str(Path(cache_root) / \\"serving_manifest.json\\"), \\"reused\\": False}\\n    _ACTIVE = {\\"server\\": server, \\"thread\\": thread, \\"adapter\\": adapter,\\n               \\"settings\\": settings, \\"info\\": info}\\n    print(\\"Private backend ready:\\", info[\\"url\\"], flush=True)\\n    return info\\n\\n\\ndef stop_backend():\\n    global _ACTIVE\\n    if _ACTIVE:\\n        _ACTIVE[\\"server\\"].shutdown()\\n        _ACTIVE[\\"server\\"].server_close()\\n        _ACTIVE[\\"thread\\"].join(timeout=5)\\n        _ACTIVE = None\\n        print(\\"Backend stopped. Stop the Colab session separately when finished.\\", flush=True)\\n", "apps/backend/requirements-model.txt": "# Only needed for --baseline-dir. PyTorch must also be installed.\\ntransformers==4.57.1\\nsafetensors\\n", "apps/backend/server.py": "\\"\\"\\"Development HTTP boundary for the screening assistant (stdlib only).\\n\\nRoutes: GET /health, GET /benchmark/claims, POST /verify.\\nSee docs/INTEGRATION_CONTRACT.md. Supports fixture replay and the saved baseline\\nadapter; other model adapters plug in through the same boundary.\\n\\"\\"\\"\\nimport argparse\\nimport json\\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\\nfrom pathlib import Path\\nfrom urllib.parse import parse_qs, urlparse\\n\\nLABELS = (\\"Supported\\", \\"Refuted\\", \\"Not Enough Evidence\\", \\"Conflicting Evidence\\")\\nMAX_EVIDENCE = 3\\nMAX_CLAIM_CHARS = 2000\\nMAX_BODY_BYTES = 64 * 1024\\nFIXTURE_PATH = Path(__file__).parent / \\"fixtures\\" / \\"demo_claims.json\\"\\n\\n\\nclass Adapter:\\n    \\"\\"\\"Verifier boundary: ordered passages in, verdict fields out.\\"\\"\\"\\n\\n    name = \\"base\\"\\n\\n    def verify(self, claim, passages):  # pragma: no cover - interface\\n        raise NotImplementedError\\n\\n\\nclass FixtureReplayAdapter(Adapter):\\n    \\"\\"\\"Replays saved fixture results. This is NOT new model inference.\\"\\"\\"\\n\\n    name = \\"fixture_replay\\"\\n\\n    def __init__(self, path=FIXTURE_PATH):\\n        data = json.loads(Path(path).read_text(encoding=\\"utf-8\\"))\\n        self.records = {r[\\"claim_id\\"]: r for r in data[\\"claims\\"]}\\n\\n    def list_claims(self, query=\\"\\", limit=50):\\n        q = (query or \\"\\").strip().lower()\\n        rows = [{\\"claim_id\\": k, \\"claim\\": v[\\"claim\\"]} for k, v in self.records.items()\\n                if q in k.lower() or q in v[\\"claim\\"].lower()]\\n        return rows[:limit]\\n\\n    def get(self, claim_id):\\n        return self.records.get(claim_id)\\n\\n\\ndef normalize_evidence(items):\\n    \\"\\"\\"Order by rank, keep at most three, renumber ranks 1..n.\\"\\"\\"\\n    ordered = sorted(items, key=lambda e: e.get(\\"rank\\", 0))[:MAX_EVIDENCE]\\n    return [\\n        {\\"passage_id\\": e[\\"passage_id\\"], \\"text\\": e[\\"text\\"], \\"url\\": e[\\"url\\"], \\"rank\\": i}\\n        for i, e in enumerate(ordered, start=1)\\n    ]\\n\\n\\ndef error_response(mode, message):\\n    return {\\"mode\\": mode, \\"status\\": \\"error\\", \\"claim\\": None, \\"verdict\\": None,\\n            \\"explanation\\": message, \\"uncertainty_note\\": None, \\"evidence\\": []}\\n\\n\\ndef handle_verify(payload, adapter):\\n    \\"\\"\\"Return (http_status, response_dict). Pure function so tests need no socket.\\"\\"\\"\\n    if not isinstance(payload, dict):\\n        return 400, error_response(None, \\"Request body must be a JSON object.\\")\\n    mode = payload.get(\\"mode\\")\\n    if mode == \\"live\\":\\n        claim = payload.get(\\"claim\\")\\n        if not isinstance(claim, str) or not claim.strip():\\n            return 400, error_response(mode, \\"A non-empty claim is required.\\")\\n        if len(claim) > MAX_CLAIM_CHARS:\\n            return 400, error_response(mode, f\\"Claim exceeds {MAX_CLAIM_CHARS} characters.\\")\\n        # No retrieval path exists for arbitrary claims. Never attach another\\n        # claim\'s evidence; report the limitation instead.\\n        return 200, {\\"mode\\": \\"live\\", \\"status\\": \\"insufficient_sources\\", \\"claim\\": claim.strip(),\\n                     \\"verdict\\": None,\\n                     \\"explanation\\": \\"Live checking is not available yet: no source retrieval is set up for new claims.\\",\\n                     \\"uncertainty_note\\": \\"Nothing was checked. Do not treat this as a result.\\",\\n                     \\"evidence\\": []}\\n    if mode == \\"benchmark\\":\\n        claim_id = payload.get(\\"claim_id\\")\\n        record = adapter.get(claim_id) if isinstance(claim_id, str) else None\\n        if record is None:\\n            return 404, error_response(mode, \\"Unknown benchmark claim_id.\\")\\n        if record.get(\\"simulate\\") == \\"error\\":\\n            return 503, error_response(mode, \\"Simulated backend failure (fixture).\\")\\n        status = record.get(\\"status\\", \\"completed\\")\\n        verdict = record.get(\\"verdict\\") if status == \\"completed\\" else None\\n        if status == \\"completed\\" and verdict not in LABELS:\\n            return 500, error_response(mode, \\"Fixture has an invalid verdict label.\\")\\n        return 200, {\\"mode\\": \\"benchmark\\", \\"status\\": status, \\"claim\\": record[\\"claim\\"],\\n                     \\"verdict\\": verdict, \\"explanation\\": record.get(\\"explanation\\"),\\n                     \\"uncertainty_note\\": record.get(\\"uncertainty_note\\"),\\n                     \\"evidence\\": normalize_evidence(record.get(\\"evidence\\", [])),\\n                     \\"provenance\\": adapter.name}\\n    return 400, error_response(mode if isinstance(mode, str) else None,\\n                               \\"mode must be \'benchmark\' or \'live\'.\\")\\n\\n\\ndef make_handler(adapter):\\n    class Handler(BaseHTTPRequestHandler):\\n        def _send(self, code, body):\\n            data = json.dumps(body, ensure_ascii=False).encode(\\"utf-8\\")\\n            self.send_response(code)\\n            self.send_header(\\"Content-Type\\", \\"application/json; charset=utf-8\\")\\n            self.send_header(\\"Content-Length\\", str(len(data)))\\n            self.send_header(\\"Access-Control-Allow-Origin\\", \\"*\\")\\n            self.send_header(\\"Access-Control-Allow-Methods\\", \\"GET, POST, OPTIONS\\")\\n            self.send_header(\\"Access-Control-Allow-Headers\\", \\"Content-Type\\")\\n            self.end_headers()\\n            self.wfile.write(data)\\n\\n        def do_OPTIONS(self):\\n            self._send(204, {})\\n\\n        def do_GET(self):\\n            if self.path == \\"/health\\":\\n                self._send(200, {\\"ok\\": True, \\"adapter\\": adapter.name})\\n            elif urlparse(self.path).path == \\"/benchmark/claims\\":\\n                params = parse_qs(urlparse(self.path).query)\\n                query = (params.get(\\"q\\") or [\\"\\"])[0][:200]\\n                try:\\n                    limit = max(1, min(1000, int((params.get(\\"limit\\") or [\\"50\\"])[0])))\\n                except ValueError:\\n                    limit = 50\\n                self._send(200, {\\"claims\\": adapter.list_claims(query, limit), \\"provenance\\": adapter.name})\\n            else:\\n                self._send(404, error_response(None, \\"Not found.\\"))\\n\\n        def do_POST(self):\\n            if self.path != \\"/verify\\":\\n                return self._send(404, error_response(None, \\"Not found.\\"))\\n            try:\\n                length = int(self.headers.get(\\"Content-Length\\", \\"0\\"))\\n            except ValueError:\\n                length = -1\\n            if length < 0 or length > MAX_BODY_BYTES:\\n                return self._send(413, error_response(None, \\"Body missing or too large.\\"))\\n            try:\\n                payload = json.loads(self.rfile.read(length).decode(\\"utf-8\\"))\\n            except (UnicodeDecodeError, json.JSONDecodeError):\\n                return self._send(400, error_response(None, \\"Invalid JSON.\\"))\\n            code, body = handle_verify(payload, adapter)\\n            self._send(code, body)\\n\\n        def log_message(self, fmt, *args):  # keep claim text out of logs\\n            pass\\n\\n    return Handler\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser()\\n    parser.add_argument(\\"--host\\", default=\\"127.0.0.1\\")\\n    parser.add_argument(\\"--port\\", type=int, default=8000)\\n    parser.add_argument(\\"--baseline-dir\\", help=\\"Saved baseline run folder (final_model/, transformer_run_config.json)\\")\\n    parser.add_argument(\\"--dev-rankings\\", help=\\"Saved dev_rankings.jsonl (BM25 top 20)\\")\\n    parser.add_argument(\\"--device\\", default=\\"cpu\\")\\n    args = parser.parse_args()\\n    if args.baseline_dir or args.dev_rankings:\\n        if not (args.baseline_dir and args.dev_rankings):\\n            parser.error(\\"--baseline-dir and --dev-rankings must be given together\\")\\n        from baseline_adapter import BaselineAdapter\\n        adapter = BaselineAdapter.from_paths(args.baseline_dir, args.dev_rankings, args.device)\\n    else:\\n        adapter = FixtureReplayAdapter()\\n    server = ThreadingHTTPServer((args.host, args.port), make_handler(adapter))\\n    print(f\\"Backend on http://{args.host}:{args.port} (adapter: {adapter.name})\\")\\n    server.serve_forever()\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "apps/browser-extension/manifest.json": "{\\n  \\"manifest_version\\": 3,\\n  \\"name\\": \\"Claim Check (course prototype)\\",\\n  \\"version\\": \\"0.1.0\\",\\n  \\"description\\": \\"Select a claim, request a check, and review preliminary verdict and sources. Prototype for CMU 10-718.\\",\\n  \\"action\\": { \\"default_title\\": \\"Check a claim\\", \\"default_popup\\": \\"popup.html\\" },\\n  \\"permissions\\": [\\"activeTab\\", \\"scripting\\", \\"clipboardRead\\"],\\n  \\"host_permissions\\": [\\"http://127.0.0.1:8000/*\\", \\"http://localhost:8000/*\\"]\\n}\\n", "apps/browser-extension/mock.js": "// Built-in SYNTHETIC samples so the UI can be exercised without a backend.\\n// These are not real checks and not model output. Same shape as the contract.\\nconst MOCK_SCENARIOS = {\\n  supported: { label: \\"Supported (2 sources)\\", status: \\"completed\\", verdict: \\"Supported\\",\\n    explanation: \\"The sample passages agree with the claim.\\", uncertainty_note: \\"Sample data only.\\",\\n    evidence: [\\n      { passage_id: \\"mock-1\\", rank: 1, url: \\"https://example.com/a\\", text: \\"Sample passage that agrees with the claim.\\" },\\n      { passage_id: \\"mock-2\\", rank: 2, url: \\"https://example.com/b\\", text: \\"A second sample passage with the same conclusion.\\" }] },\\n  refuted: { label: \\"Refuted (3 sources)\\", status: \\"completed\\", verdict: \\"Refuted\\",\\n    explanation: \\"The sample passages contradict the claim.\\", uncertainty_note: \\"Sample data only.\\",\\n    evidence: [1, 2, 3].map(i => ({ passage_id: \\"mock-r\\" + i, rank: i, url: \\"https://example.com/r\\" + i,\\n      text: \\"Sample passage \\" + i + \\" that contradicts the claim.\\" })) },\\n  nei: { label: \\"Not Enough Evidence\\", status: \\"completed\\", verdict: \\"Not Enough Evidence\\",\\n    explanation: \\"The sample passages do not settle the claim.\\", uncertainty_note: \\"Additional sources would be needed.\\",\\n    evidence: [{ passage_id: \\"mock-n1\\", rank: 1, url: \\"https://example.com/n\\", text: \\"Related but inconclusive sample passage.\\" }] },\\n  conflicting: { label: \\"Conflicting Evidence\\", status: \\"completed\\", verdict: \\"Conflicting Evidence\\",\\n    explanation: \\"One sample passage supports the claim and another contradicts it.\\", uncertainty_note: \\"The sources disagree; read both.\\",\\n    evidence: [\\n      { passage_id: \\"mock-c1\\", rank: 1, url: \\"https://example.com/c1\\", text: \\"Sample passage in favour.\\" },\\n      { passage_id: \\"mock-c2\\", rank: 2, url: \\"https://example.com/c2\\", text: \\"Sample passage against.\\" }] },\\n  insufficient: { label: \\"No sources found\\", status: \\"insufficient_sources\\", verdict: null,\\n    explanation: \\"No usable source passages were found for this claim.\\", uncertainty_note: null, evidence: [] },\\n  error: { label: \\"Service error\\", status: \\"error\\", verdict: null,\\n    explanation: \\"The service could not complete the check.\\", uncertainty_note: null, evidence: [] },\\n};\\n\\nasync function mockVerify(claim, scenarioKey) {\\n  await new Promise(r => setTimeout(r, 700)); // visible loading state\\n  return Object.assign({ mode: \\"mock\\", claim: claim }, MOCK_SCENARIOS[scenarioKey]);\\n}\\n", "apps/browser-extension/popup.css": ":root {\\n  --bg: #ffffff; --fg: #1d2330; --muted: #5b6475; --line: #e1e4ea; --card: #f6f7f9;\\n  --accent: #2f5bd8; --accent-fg: #ffffff;\\n  --supported: #17713a; --refuted: #a8271f; --nei: #5b6475; --conflict: #8a5a00; --warn: #8a5a00;\\n}\\n@media (prefers-color-scheme: dark) {\\n  :root { --bg: #171a21; --fg: #e8eaf0; --muted: #a3abbb; --line: #333a49; --card: #20242e;\\n    --accent: #7d9bff; --accent-fg: #101319;\\n    --supported: #5fd08a; --refuted: #ff8f86; --nei: #a3abbb; --conflict: #f0b84d; --warn: #f0b84d; }\\n}\\n* { box-sizing: border-box; }\\nbody { width: 400px; margin: 0; padding: 14px 14px 12px; background: var(--bg); color: var(--fg);\\n  font: 13px/1.45 system-ui, -apple-system, \\"Segoe UI\\", sans-serif; }\\n\\nheader { margin-bottom: 10px; }\\nh1 { display: flex; align-items: center; gap: 8px; font-size: 16px; margin: 0; letter-spacing: -.01em; }\\n.logo { display: inline-grid; place-items: center; width: 22px; height: 22px; border-radius: 7px;\\n  background: var(--accent); color: var(--accent-fg); font-size: 13px; font-weight: 800; }\\n.sub { color: var(--muted); margin: 3px 0 0; font-size: 12px; }\\n.hint { color: var(--muted); margin: 6px 0 8px; font-size: 12px; min-height: 16px; }\\n\\n/* Chrome caps popups near 600px tall: once a result is showing, drop the explanatory lines. */\\nbody:has(#result:not(:empty)) .sub, body:has(#result:not(:empty)) .hint { display: none; }\\nbody:has(#result:not(:empty)) textarea { min-height: 44px; }\\n\\n/* segmented mode control */\\n.seg { display: flex; gap: 2px; padding: 3px; background: var(--card); border: 1px solid var(--line); border-radius: 10px; }\\n.seg button { flex: 1; padding: 5px 8px; border: 0; border-radius: 7px; background: transparent;\\n  color: var(--muted); font: inherit; font-weight: 600; cursor: pointer; }\\n.seg button[aria-checked=\\"true\\"] { background: var(--bg); color: var(--fg);\\n  box-shadow: 0 1px 2px rgba(0, 0, 0, .18), 0 0 0 1px var(--line); }\\n\\nlabel { display: block; font-weight: 600; margin: 8px 0 3px; font-size: 12px; }\\nselect, textarea, input:not([type=\\"hidden\\"]) { width: 100%; padding: 7px 9px; border: 1px solid var(--line);\\n  border-radius: 8px; background: var(--bg); color: var(--fg); font: inherit; }\\n#search { margin-bottom: 6px; }\\ntextarea { resize: vertical; min-height: 58px; }\\ntextarea:read-only { background: var(--card); }\\n\\n.tools { display: flex; align-items: center; gap: 6px; margin: 4px 0 10px; color: var(--muted); font-size: 12px; }\\nbutton.link { padding: 0; border: 0; background: none; color: var(--accent); font: inherit; font-size: 12px;\\n  cursor: pointer; }\\nbutton.link:hover { text-decoration: underline; }\\nbutton.primary { width: 100%; padding: 9px 12px; border-radius: 8px; border: 0; background: var(--accent);\\n  color: var(--accent-fg); font: inherit; font-weight: 700; cursor: pointer; }\\nbutton.primary:hover:not(:disabled) { filter: brightness(1.07); }\\nbutton:disabled { opacity: .55; cursor: default; }\\n:focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; }\\n\\n#result { margin-top: 12px; }\\n#result:empty { margin: 0; }\\n.card { background: var(--card); border: 1px solid var(--line); border-radius: 10px; padding: 10px 12px; }\\n.badge { display: inline-block; font-size: 11px; padding: 1px 8px; border: 1px solid var(--line);\\n  border-radius: 999px; color: var(--muted); margin-bottom: 8px; background: var(--bg); }\\n.note { color: var(--muted); margin: 6px 0 0; font-size: 12px; }\\n\\n/* verdict banner */\\n.card.verdict-card { --vc: var(--nei); border-left: 4px solid var(--vc);\\n  background: color-mix(in srgb, var(--vc) 9%, var(--bg)); }\\n.verdict-card.Supported { --vc: var(--supported); }\\n.verdict-card.Refuted { --vc: var(--refuted); }\\n.verdict-card.NEI { --vc: var(--nei); }\\n.verdict-card.Conflicting { --vc: var(--conflict); }\\n.verdict { display: flex; align-items: center; gap: 8px; color: var(--vc); font-size: 17px; font-weight: 800; margin: 0 0 4px; }\\n.verdict .icon { display: inline-grid; place-items: center; width: 22px; height: 22px; border-radius: 50%;\\n  background: var(--vc); color: var(--bg); font-size: 13px; line-height: 1; }\\n.claimline { color: var(--muted); font-size: 12px; margin: 0 0 6px; }\\n\\n.msg-warn { border-color: var(--warn); }\\n.msg-warn .title { color: var(--warn); font-weight: 700; }\\n.msg-error { border-color: var(--refuted); }\\n.msg-error .title { color: var(--refuted); font-weight: 700; }\\n\\nh2 { font-size: 11px; margin: 12px 0 6px; color: var(--muted); text-transform: uppercase; letter-spacing: .06em; }\\nol { list-style: none; margin: 0; padding: 0; }\\ndetails.src { border: 1px solid var(--line); border-radius: 10px; margin-bottom: 6px; background: var(--bg); }\\ndetails.src summary { list-style: none; cursor: pointer; padding: 8px 10px; }\\ndetails.src summary::-webkit-details-marker { display: none; }\\n.src-head { display: flex; align-items: center; gap: 6px; font-size: 11px; color: var(--muted); }\\n.src-rank { display: inline-grid; place-items: center; width: 18px; height: 18px; border-radius: 50%;\\n  background: var(--accent); color: var(--accent-fg); font-weight: 700; }\\n.src-domain { font-weight: 600; color: var(--fg); overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }\\n.src-chev { margin-left: auto; transition: transform .15s; }\\ndetails.src[open] .src-chev { transform: rotate(90deg); }\\n.preview { margin-top: 4px; display: -webkit-box; -webkit-line-clamp: 2; -webkit-box-orient: vertical; overflow: hidden; }\\ndetails.src[open] .preview { display: none; }\\n.src-body { padding: 0 10px 10px; }\\n.src-body blockquote { margin: 0 0 6px; padding-left: 8px; border-left: 3px solid var(--line); max-height: 180px; overflow: auto; }\\n.src-body a { color: var(--accent); word-break: break-all; font-size: 12px; }\\n\\ndetails.settings { margin-top: 12px; }\\ndetails.settings summary { cursor: pointer; color: var(--muted); font-size: 12px; }\\n\\n.spinner { display: inline-block; width: 12px; height: 12px; margin-right: 6px; vertical-align: -2px;\\n  border: 2px solid var(--line); border-top-color: var(--accent); border-radius: 50%;\\n  animation: spin .8s linear infinite; }\\n@keyframes spin { to { transform: rotate(360deg); } }\\n@media (prefers-reduced-motion: reduce) { .spinner { animation: none; } .src-chev { transition: none; } }\\n", "apps/browser-extension/popup.html": "<!doctype html>\\n<html lang=\\"en\\">\\n<head>\\n  <meta charset=\\"utf-8\\">\\n  <meta name=\\"viewport\\" content=\\"width=device-width, initial-scale=1\\">\\n  <title>Claim Check</title>\\n  <link rel=\\"stylesheet\\" href=\\"popup.css\\">\\n</head>\\n<body>\\n  <header>\\n    <h1><span class=\\"logo\\" aria-hidden=\\"true\\">\\u2713</span>Claim Check</h1>\\n    <p class=\\"sub\\">A preliminary screen, not a final answer. You decide what to trust or share.</p>\\n  </header>\\n\\n  <section>\\n    <input type=\\"hidden\\" id=\\"mode\\" value=\\"mock\\">\\n    <div class=\\"seg\\" role=\\"radiogroup\\" aria-label=\\"Source of results\\">\\n      <button type=\\"button\\" role=\\"radio\\" data-mode=\\"mock\\" aria-checked=\\"true\\">Mock</button>\\n      <button type=\\"button\\" role=\\"radio\\" data-mode=\\"benchmark\\" aria-checked=\\"false\\">Benchmark</button>\\n      <button type=\\"button\\" role=\\"radio\\" data-mode=\\"live\\" aria-checked=\\"false\\">Live</button>\\n    </div>\\n    <p id=\\"modeHint\\" class=\\"hint\\"></p>\\n\\n    <div id=\\"scenarioRow\\" hidden>\\n      <label for=\\"scenario\\" id=\\"scenarioLabel\\">Sample outcome</label>\\n      <input id=\\"search\\" type=\\"search\\" placeholder=\\"Search demo claims\\" hidden>\\n      <select id=\\"scenario\\"></select>\\n    </div>\\n\\n    <label for=\\"claim\\">Claim</label>\\n    <textarea id=\\"claim\\" rows=\\"3\\" maxlength=\\"2000\\" placeholder=\\"Select text on a page, or type or paste a claim\\"></textarea>\\n    <div class=\\"tools\\">\\n      <button id=\\"useSelection\\" type=\\"button\\" class=\\"link\\">Use selected text</button>\\n      <span aria-hidden=\\"true\\">\\u00b7</span>\\n      <button id=\\"pasteClip\\" type=\\"button\\" class=\\"link\\">Paste from clipboard</button>\\n    </div>\\n    <button id=\\"check\\" type=\\"button\\" class=\\"primary\\">Check this claim</button>\\n  </section>\\n\\n  <section id=\\"result\\" aria-live=\\"polite\\"></section>\\n\\n  <details class=\\"settings\\">\\n    <summary>Backend settings</summary>\\n    <label for=\\"backend\\">Backend URL</label>\\n    <input id=\\"backend\\" value=\\"http://127.0.0.1:8000\\">\\n  </details>\\n\\n  <script src=\\"mock.js\\"></script>\\n  <script src=\\"popup.js\\"></script>\\n</body>\\n</html>\\n", "apps/browser-extension/popup.js": "\\"use strict\\";\\n\\nconst $ = id => document.getElementById(id);\\nconst els = { search: $(\\"search\\"), mode: $(\\"mode\\"), hint: $(\\"modeHint\\"), scenarioRow: $(\\"scenarioRow\\"), scenario: $(\\"scenario\\"),\\n  claim: $(\\"claim\\"), check: $(\\"check\\"), useSel: $(\\"useSelection\\"), result: $(\\"result\\"), backend: $(\\"backend\\") };\\n\\nconst HINTS = {\\n  mock: \\"Built-in samples for trying the interface. Not a real check.\\",\\n  benchmark: \\"Known benchmark claims only, served by the local backend. Not a check of arbitrary text.\\",\\n  live: \\"Checking new claims is not available yet; you will see an explanation instead of a result.\\",\\n};\\nconst MODE_BADGE = { mock: \\"Mock sample\\", benchmark: \\"Benchmark replay (saved)\\", live: \\"Live\\" };\\n// The backend says where a benchmark result came from; label it truthfully.\\nconst PROVENANCE_BADGE = {\\n  fixture_replay: \\"Benchmark replay (synthetic sample)\\",\\n  baseline_distilbert_bm25_top3: \\"Baseline model, benchmark claim (not a live check)\\",\\n};\\nconst VERDICT_CLASS = { \\"Supported\\": \\"Supported\\", \\"Refuted\\": \\"Refuted\\",\\n  \\"Not Enough Evidence\\": \\"NEI\\", \\"Conflicting Evidence\\": \\"Conflicting\\" };\\nconst VERDICT_ICON = { \\"Supported\\": \\"\\u2713\\", \\"Refuted\\": \\"\\u2715\\", \\"Not Enough Evidence\\": \\"?\\", \\"Conflicting Evidence\\": \\"\\u2194\\" };\\nconst MAX_SOURCES = 3;\\n\\nfunction domainOf(href) { try { return new URL(href).hostname.replace(/^www\\\\./, \\"\\"); } catch { return href; } }\\n\\nlet benchmarkClaims = []; // [{claim_id, claim}]\\n\\nfunction el(tag, cls, text) {\\n  const n = document.createElement(tag);\\n  if (cls) n.className = cls;\\n  if (text != null) n.textContent = text; // never innerHTML: claim/source text is untrusted\\n  return n;\\n}\\n\\nfunction safeHttpUrl(u) {\\n  try { const p = new URL(u); return (p.protocol === \\"https:\\" || p.protocol === \\"http:\\") ? p.href : null; }\\n  catch { return null; }\\n}\\n\\nfunction show(node) { els.result.replaceChildren(node); }\\n\\nfunction renderLoading() {\\n  const c = el(\\"div\\", \\"card\\");\\n  c.append(el(\\"span\\", \\"spinner\\"), document.createTextNode(\\"Checking\\u2026\\"));\\n  show(c);\\n}\\n\\nfunction renderMessage(kind, title, body, badge) {\\n  const c = el(\\"div\\", \\"card msg-\\" + kind);\\n  if (badge) c.append(el(\\"div\\", \\"badge\\", badge));\\n  c.append(el(\\"div\\", \\"title\\", title), el(\\"p\\", \\"note\\", body));\\n  show(c);\\n}\\n\\nfunction renderResponse(r) {\\n  const badge = PROVENANCE_BADGE[r.provenance] || MODE_BADGE[r.mode] || \\"\\";\\n  if (r.status === \\"insufficient_sources\\") {\\n    return renderMessage(\\"warn\\", \\"Not enough sources to check this\\",\\n      (r.explanation || \\"\\") + (r.uncertainty_note ? \\" \\" + r.uncertainty_note : \\"\\"), badge);\\n  }\\n  if (r.status !== \\"completed\\" || !r.verdict) {\\n    return renderMessage(\\"error\\", \\"Something went wrong\\", r.explanation || \\"The check could not be completed.\\", badge);\\n  }\\n  const wrap = el(\\"div\\");\\n  const card = el(\\"div\\", \\"card verdict-card \\" + (VERDICT_CLASS[r.verdict] || \\"\\"));\\n  card.append(el(\\"div\\", \\"badge\\", badge));\\n  // The claim box already shows the claim in benchmark mode; repeat it only if it differs.\\n  if (r.claim && r.claim !== els.claim.value.trim()) card.append(el(\\"p\\", \\"claimline\\", \\"Claim checked: \\" + r.claim));\\n  const v = el(\\"div\\", \\"verdict\\");\\n  v.append(el(\\"span\\", \\"icon\\", VERDICT_ICON[r.verdict] || \\"\\u2022\\"), document.createTextNode(r.verdict));\\n  card.append(v, el(\\"div\\", null, r.explanation || \\"\\"));\\n  if (r.uncertainty_note) card.append(el(\\"p\\", \\"note\\", r.uncertainty_note));\\n  card.append(el(\\"p\\", \\"note\\", \\"Preliminary screening only. Read the sources and decide for yourself.\\"));\\n  wrap.append(card);\\n\\n  const sources = (r.evidence || []).slice().sort((a, b) => a.rank - b.rank).slice(0, MAX_SOURCES);\\n  if (sources.length) {\\n    wrap.append(el(\\"h2\\", null, \\"Sources, most relevant first\\"));\\n    const list = el(\\"ol\\");\\n    sources.forEach((s, i) => {\\n      const href = safeHttpUrl(s.url);\\n      const li = el(\\"li\\");\\n      const d = el(\\"details\\", \\"src\\");\\n      const sum = el(\\"summary\\");\\n      const head = el(\\"div\\", \\"src-head\\");\\n      head.append(el(\\"span\\", \\"src-rank\\", String(i + 1)), el(\\"span\\", \\"src-domain\\", href ? domainOf(href) : \\"Source link unavailable\\"),\\n        el(\\"span\\", \\"src-chev\\", \\"\\u203a\\"));\\n      sum.append(head, el(\\"div\\", \\"preview\\", s.text));\\n      const body = el(\\"div\\", \\"src-body\\");\\n      body.append(el(\\"blockquote\\", null, s.text));\\n      if (href) {\\n        const a = el(\\"a\\", null, href);\\n        a.href = href; a.target = \\"_blank\\"; a.rel = \\"noopener noreferrer\\";\\n        body.append(a);\\n      }\\n      d.append(sum, body);\\n      li.append(d);\\n      list.append(li);\\n    });\\n    wrap.append(list);\\n  }\\n  show(wrap);\\n}\\n\\nasync function postVerify(body) {\\n  const base = els.backend.value.replace(/\\\\/+$/, \\"\\");\\n  let resp;\\n  try {\\n    resp = await fetch(base + \\"/verify\\", { method: \\"POST\\", headers: { \\"Content-Type\\": \\"application/json\\" },\\n      body: JSON.stringify(body) });\\n  } catch {\\n    return { mode: body.mode, status: \\"error\\", verdict: null, evidence: [],\\n      explanation: \\"Could not reach the backend at \\" + base + \\". Is it running?\\" };\\n  }\\n  try { return await resp.json(); }\\n  catch { return { mode: body.mode, status: \\"error\\", verdict: null, evidence: [], explanation: \\"The backend sent an unreadable reply.\\" }; }\\n}\\n\\nasync function runCheck() {\\n  const mode = els.mode.value;\\n  const claim = els.claim.value.trim();\\n  if (!claim) return renderMessage(\\"warn\\", \\"Enter a claim first\\", \\"Select text on a page or type one above.\\");\\n  els.check.disabled = true;\\n  renderLoading();\\n  try {\\n    let r;\\n    if (mode === \\"mock\\") r = await mockVerify(claim, els.scenario.value);\\n    else if (mode === \\"live\\") r = await postVerify({ mode: \\"live\\", claim, page_url: null });\\n    else {\\n      const picked = benchmarkClaims.find(c => c.claim_id === els.scenario.value);\\n      r = picked ? await postVerify({ mode: \\"benchmark\\", claim_id: picked.claim_id })\\n        : { mode, status: \\"error\\", verdict: null, evidence: [], explanation: \\"Choose a demo claim.\\" };\\n    }\\n    renderResponse(r);\\n    els.result.scrollIntoView({ block: \\"start\\", behavior: \\"smooth\\" }); // popup is short: bring the verdict into view\\n  } finally { els.check.disabled = false; }\\n}\\n\\nasync function loadBenchmarkClaims() {\\n  els.scenario.replaceChildren();\\n  let reachable = true;\\n  try {\\n    const base = els.backend.value.replace(/\\\\/+$/, \\"\\");\\n    const q = encodeURIComponent(els.search.value.trim());\\n    const data = await (await fetch(base + \\"/benchmark/claims?limit=50&q=\\" + q)).json();\\n    benchmarkClaims = data.claims || [];\\n  } catch { benchmarkClaims = []; reachable = false; }\\n  if (!reachable) {\\n    renderMessage(\\"error\\", \\"Backend not reachable\\", \\"Start it with: python3 apps/backend/server.py\\", MODE_BADGE.benchmark);\\n    return;\\n  }\\n  if (!benchmarkClaims.length) {\\n    els.claim.value = \\"\\";\\n    renderMessage(\\"warn\\", \\"No demo claims match\\", \\"Try a different search.\\");\\n    return;\\n  }\\n  benchmarkClaims.forEach(c => els.scenario.append(new Option(c.claim_id + \\" \\u00b7 \\" + c.claim.slice(0, 60), c.claim_id)));\\n  els.claim.value = benchmarkClaims[0].claim;\\n  els.result.replaceChildren();\\n}\\n\\nfunction applyMode() {\\n  const mode = els.mode.value;\\n  els.hint.textContent = HINTS[mode];\\n  els.scenarioRow.hidden = mode === \\"live\\";\\n  els.search.hidden = mode !== \\"benchmark\\";\\n  els.claim.readOnly = mode === \\"benchmark\\"; // benchmark replays a fixed demo claim only\\n  els.claim.placeholder = mode === \\"benchmark\\" ? \\"Pick a demo claim above\\"\\n    : \\"Select text on a page, or type or paste a claim\\";\\n  $(\\"scenarioLabel\\").textContent = mode === \\"mock\\" ? \\"Sample outcome\\" : \\"Demo claim\\";\\n  els.result.replaceChildren();\\n  if (mode === \\"mock\\") {\\n    els.scenario.replaceChildren(...Object.entries(MOCK_SCENARIOS).map(([k, v]) => new Option(v.label, k)));\\n  } else if (mode === \\"benchmark\\") {\\n    loadBenchmarkClaims();\\n  }\\n}\\n\\nconst NO_SELECTION_HELP = \\"Nothing could be read from the page. Some sites (for example Google Docs) \\" +\\n  \\"don\'t expose selected text to extensions. Copy the text, then use Paste from clipboard.\\";\\n\\nfunction benchmarkLocked() {\\n  if (els.mode.value !== \\"benchmark\\") return false;\\n  renderMessage(\\"warn\\", \\"Demo claims only\\", \\"Benchmark demo replays saved results for the listed demo claims. Switch to Mock or Live to use your own text.\\");\\n  return true;\\n}\\n\\nasync function useSelection(quiet) {\\n  if (els.mode.value === \\"benchmark\\") { if (!quiet) benchmarkLocked(); return; }\\n  if (typeof chrome === \\"undefined\\" || !chrome.tabs || !chrome.scripting) return;\\n  try {\\n    const [tab] = await chrome.tabs.query({ active: true, currentWindow: true });\\n    const results = await chrome.scripting.executeScript({ target: { tabId: tab.id, allFrames: true },\\n      func: () => String(window.getSelection()) });\\n    const text = results.map(r => (r.result || \\"\\").trim()).sort((a, b) => b.length - a.length)[0] || \\"\\";\\n    if (text) els.claim.value = text.slice(0, 2000);\\n    else if (!quiet) renderMessage(\\"warn\\", \\"No text selected\\", NO_SELECTION_HELP);\\n  } catch {\\n    if (!quiet) renderMessage(\\"warn\\", \\"Can\'t read this page\\", \\"Browser pages and the web store block this. Paste the claim instead.\\");\\n  }\\n}\\n\\nasync function pasteClipboard() {\\n  if (benchmarkLocked()) return;\\n  try {\\n    const text = (await navigator.clipboard.readText()).trim();\\n    if (text) { els.claim.value = text.slice(0, 2000); els.result.replaceChildren(); }\\n    else renderMessage(\\"warn\\", \\"Clipboard is empty\\", \\"Copy the claim text first.\\");\\n  } catch {\\n    renderMessage(\\"warn\\", \\"Clipboard not available\\", \\"Click in the Claim box and paste with Cmd/Ctrl+V.\\");\\n  }\\n}\\n\\ndocument.querySelectorAll(\\".seg button\\").forEach(b => b.addEventListener(\\"click\\", () => {\\n  if (els.mode.value === b.dataset.mode) return;\\n  els.mode.value = b.dataset.mode;\\n  document.querySelectorAll(\\".seg button\\").forEach(x => x.setAttribute(\\"aria-checked\\", String(x === b)));\\n  applyMode();\\n}));\\nlet searchTimer;\\nels.search.addEventListener(\\"input\\", () => { clearTimeout(searchTimer); searchTimer = setTimeout(loadBenchmarkClaims, 250); });\\nels.scenario.addEventListener(\\"change\\", () => {\\n  if (els.mode.value !== \\"benchmark\\") return;\\n  const c = benchmarkClaims.find(x => x.claim_id === els.scenario.value);\\n  if (c) els.claim.value = c.claim;\\n});\\nels.check.addEventListener(\\"click\\", runCheck);\\nels.useSel.addEventListener(\\"click\\", () => useSelection(false));\\n$(\\"pasteClip\\").addEventListener(\\"click\\", pasteClipboard);\\napplyMode();\\nuseSelection(true); // quietly prefill if text is already selected\\n"}')
SOURCE_SHA256 = '1741ca0c69eb2766e444954781fbf45cbc2e6ab0c2e7b18ceae10d08f0ebb334'
assert hashlib.sha256(json.dumps(SOURCES, sort_keys=True, ensure_ascii=True).encode()).hexdigest() == SOURCE_SHA256
RUNTIME = Path('/content') / ('claim_check_source_' + SOURCE_SHA256[:12])
for name, module in list(sys.modules.items()):
    if name.startswith('claim_check_colab_runtime_') and name != 'claim_check_colab_runtime_' + SOURCE_SHA256[:12] and getattr(module, '_ACTIVE', None):
        raise RuntimeError('A different backend source is still loaded. Use a fresh CPU session or stop the previous backend first.')
for name, content in SOURCES.items():
    path = RUNTIME / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
print('Portable source:', SOURCE_SHA256)


## 2. Install the text-only model dependencies

In [ ]:
if 'transformers' in sys.modules and sys.modules['transformers'].__version__ != '4.57.1':
    raise RuntimeError('Use a fresh session before loading this notebook.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(RUNTIME / 'apps/backend/requirements-model.txt')], check=True)
import torch
print('PyTorch:', torch.__version__, '| inference device: CPU')


## 3. Mount Drive and locate the serving inputs

Keep the shared shortcut named `10718_averitec`. A CLI-mounted Drive is reused. Only the model, tokenizer, run settings and saved dev rankings are read; no training labels or full passage corpus are needed.

In [ ]:
if not Path('/content/drive/MyDrive').is_dir():
    from google.colab import drive
    drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/10718_averitec/v2_source_documents')
CACHE_ROOT = Path('/content/claim_check_serving')
PORT = 8000
CPU_THREADS = 2
if not DATA_ROOT.is_dir():
    raise FileNotFoundError('Check shared-folder access and the My Drive shortcut: ' + str(DATA_ROOT))
print('Shared input root:', DATA_ROOT)


## 4. Copy the small serving bundle and start the private backend

Complete cached files are reused after interruption. Shared Drive files are never changed. This returns to the kernel after starting a background HTTP thread; there is no blocking server loop or artificial keepalive. Rerunning with the same source/settings reuses the loaded backend.

In [ ]:
module_name = 'claim_check_colab_runtime_' + SOURCE_SHA256[:12]
if module_name not in sys.modules:
    spec = importlib.util.spec_from_file_location(module_name, RUNTIME / 'apps/backend/colab_runtime.py')
    backend = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = backend
    spec.loader.exec_module(backend)
else:
    backend = sys.modules[module_name]
BACKEND_INFO = backend.start_backend(DATA_ROOT, CACHE_ROOT, port=PORT, cpu_threads=CPU_THREADS)
print(json.dumps(BACKEND_INFO, indent=2))


## 5. Check one real inference request

In [ ]:
import urllib.request
opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
base = BACKEND_INFO['url']
with opener.open(base + '/health', timeout=10) as reply:
    health = json.load(reply)
assert health['adapter'] == 'baseline_distilbert_bm25_top3', health
with opener.open(base + '/benchmark/claims?limit=1', timeout=10) as reply:
    claim_id = json.load(reply)['claims'][0]['claim_id']
request = urllib.request.Request(base + '/verify',
    data=json.dumps({'mode': 'benchmark', 'claim_id': claim_id}).encode(),
    headers={'Content-Type': 'application/json'})
started = time.monotonic()
with opener.open(request, timeout=90) as reply:
    result = json.load(reply)
assert result['status'] in ('completed', 'insufficient_sources'), result
print('Claim:', claim_id, '| status:', result['status'], '| verdict:', result.get('verdict'))
print('Evidence items:', len(result['evidence']), '| runtime inference seconds:', round(time.monotonic() - started, 2))
print('Backend is ready for the local relay. No gold verdict labels were read.')


## 6. Optional: package this matching extension version

The zip contains extension code only, not Drive files, weights, credentials or predictions. Download with the Colab file browser, extract locally, then use Chrome's Load unpacked on the extracted folder.

In [ ]:
import zipfile
extension_zip = Path('/content/claim_check_extension.zip')
with zipfile.ZipFile(extension_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted((RUNTIME / 'apps/browser-extension').iterdir()):
        if path.is_file():
            archive.write(path, path.name)
print('Extension zip:', extension_zip)
print('Source snapshot:', SOURCE_SHA256)


## Connect and stop

On your computer, run the repository's `scripts/colab_extension.ps1 -Action Relay`
or the Python relay command in `apps/COLAB_BACKEND.md`. Keep that terminal open.
Load the unpacked extension, use **Benchmark**, and keep Backend URL at
`http://127.0.0.1:8000`. Mock mode does not call the trained model; Live is unavailable.

Run `colab sessions` if you opened this notebook in the browser; select that same
session in the relay, rather than creating another one. CLI command startup adds
transport latency to the first verification of each claim; repeated inference is
cached in the remote adapter. Record this overhead if measuring decision time.

If the kernel restarts, rerun the notebook in order. A replacement VM needs a new
Drive mount and cache copy. Changing settings while the backend is loaded requires
`backend.stop_backend()` first. A new notebook source needs a fresh runtime.

When finished, stop the relay with Ctrl+C and stop the named Colab runtime:
`colab stop -s claim-check-extension`. Closing the extension does not stop the VM.